# gpOmol — WL + gp2Scale GP

Runs the `wl_gp2scale` pipeline end-to-end: load OMol25 -> WL featurize -> PLS(10)
-> Wendland gp2Scale GP -> predict. Run the cells top to bottom.

**Prerequisite (one terminal command this notebook cannot run for you):** in a
terminal *inside the same allocation*, launch the Dask cluster —

```bash
./launch-dask-conda.sh 16 > dask_launch.log 2>&1 &
grep -c 'Register worker' dask_launch.log   # wait until this reads 16
```

The number (16) must match `N_WORKERS` below and your `salloc -n`. Then run this
notebook (kernel: `gpomol`, started from the repo root).

## 1. Imports

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from wl_gp2scale.data import get_data
from wl_gp2scale.pipeline import (
    WLGPPipeline, connect_dask, build_gp, predict,
    with_category_tag, sort_by_category,
)

## 2. Parameters
Everything you tune lives here, so nothing gets forgotten. For the 16k pct=10
baseline set `N = 20_000`; for the full run `N = 200_000`.

In [ ]:
N          = 200_000   # molecules to draw
CUTOFF_PCT = 10.0      # cutoff percentile (10 fits the 4h window; 25 = signal-optimal)
MIN_COUNT  = 2         # WL vocab pruning (2 matches gp_parity.py)
TEST_SIZE  = 0.02      # held-out fraction
BATCH_SIZE = 10_000    # gp2Scale block size
N_WORKERS  = 16        # MUST match ./launch-dask-conda.sh and salloc -n
VARIANCE   = False     # False = mean only (variance is 1 solve PER test point)

## 3. Connect to the Dask cluster
Reads `$SCRATCH/scheduler_file_gpOmol.json`. If this errors about a *stale* file,
you forgot to relaunch `./launch-dask-conda.sh` in this allocation.

In [ ]:
client = connect_dask(n_workers=N_WORKERS)
client

## 4. Load data
Frozen subset + descriptor-independent residual target `y = E - m(x)`, plus the
integer `data_id` category used for block-sparsity.

In [ ]:
ds = get_data(n=N, seed=0)
idx = np.arange(len(ds))
tr, te = train_test_split(idx, test_size=TEST_SIZE, random_state=42)
atoms_tr = [ds.atoms[i] for i in tr]; atoms_te = [ds.atoms[i] for i in te]
y_tr, y_te = ds.y[tr], ds.y[te]
cat_tr, cat_te = ds.data_id[tr], ds.data_id[te]
print(f'train={len(tr):,}  test={len(te):,}')

## 5. Featurize + reduce (fit on train)
WL vocab + supervised PLS are fit on train only; the cutoff is recalibrated on the
train embedding. Parallelized over the cluster.

In [ ]:
pipe = WLGPPipeline(min_count=MIN_COUNT, cutoff_percentile=CUTOFF_PCT)
Z_tr = pipe.fit(atoms_tr, y_tr, cat_tr, client=client)
Z_te = pipe.transform(atoms_te, client=client)
cutoff, dim = pipe.cutoff_, pipe.dim_
print(f'cutoff={cutoff:.5f}  dim={dim}')

## 6. Build the gp2Scale GP
Tag with category, sort into contiguous category blocks, then construct. The
constructor does the distributed assembly + the (driver-side) solve — the slow step.

In [ ]:
X_tr = with_category_tag(Z_tr, cat_tr)
X_te = with_category_tag(Z_te, cat_te)
X_tr, y_tr_sorted, order = sort_by_category(X_tr, y_tr)

import time; t0 = time.time()
gp, kern = build_gp(X_tr, y_tr_sorted, cutoff, dim, client,
                    batch_size=BATCH_SIZE, device='cuda')
print(f'GP constructed in {time.time()-t0:.0f}s')

## 7. Predict + score

In [ ]:
m, v = predict(gp, X_te, batch=500, variance=VARIANCE, verbose=True)
rmse = float(np.sqrt(np.mean((m - y_te) ** 2)))
r2 = float(r2_score(y_te, m))
print(f'TEST  R2={r2:.4f}  RMSE={rmse:.4f}  (baseline std {np.std(y_te):.4f})')

## 8. Parity plot

In [ ]:
import matplotlib.pyplot as plt
lo, hi = float(min(y_te.min(), m.min())), float(max(y_te.max(), m.max()))
fig, ax = plt.subplots(figsize=(6, 6))
c = np.sqrt(v) if VARIANCE else None
sc = ax.scatter(y_te, m, c=c, s=8, alpha=0.6, cmap='viridis')
if VARIANCE: fig.colorbar(sc, ax=ax, label='posterior std')
ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5)
ax.set_xlabel('true residual  y = E - m(x)'); ax.set_ylabel('predicted residual')
ax.set_title(f'N_train={len(tr):,}  R2={r2:.4f}'); plt.show()

## 9. Recover physical energy (optional)
`E_total = y_pred + m(x)` using the extensive mean fit in step 4.

In [ ]:
E_pred = m + ds.mean.predict(
    [ds.atoms[i].get_atomic_numbers().tolist() for i in te],
    [float(ds.atoms[i].info.get('charge', 0)) for i in te],
    [float(ds.atoms[i].info.get('spin', 1)) for i in te],
)
print('E_total predictions:', E_pred[:5])